In [9]:
import chromadb
import pandas as pd
from langchain.docstore.document import Document
client = chromadb.PersistentClient(path="chroma-data")
collection = client.get_collection(name="eidc-metadata")
result = collection.get()
titles = [metadata["dataset_title"] for metadata in result["metadatas"]]

docs = [Document(page_content=result["documents"][i], metadata=result["metadatas"][i]) for i in range(len(result["documents"]))]
for doc in docs:
    doc.metadata["filename"] = doc.metadata["dataset_id"]
    doc.metadata["source"] = doc.metadata["dataset_id"]
len(docs)

1915

In [13]:
from langchain_community.embeddings import OllamaEmbeddings
from langchain_community.chat_models import ChatOllama
llm = ChatOllama(model='mistral-nemo', num_ctx=16384)
embeddings = OllamaEmbeddings(model='mistral-nemo', num_ctx=16384)

In [11]:
from ragas.testset.generator import TestsetGenerator
from ragas.testset.evolutions import simple, reasoning, multi_context
from ragas.run_config import RunConfig
gen = TestsetGenerator.from_langchain(llm, llm, embeddings, run_config=RunConfig(max_workers=1, max_retries=1))
dist = {simple: 1, multi_context: 0.0, reasoning: 0.0}

In [12]:
import logging
import nest_asyncio
nest_asyncio.apply()
#logging.basicConfig(level=logging.INFO)
testset = gen.generate_with_langchain_docs(docs, 100, dist, is_async=False)

embedding nodes:   0%|          | 0/3830 [00:00<?, ?it/s]

Filename and doc_id are the same for all nodes.


Generating:   0%|          | 0/100 [00:00<?, ?it/s]

In [14]:
df = testset.to_pandas()
df.to_csv("ragas-testset.csv", index=False)

In [15]:
df

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What was the average weed abundance across the...,"[The dataset entitled ""Abundance of weeds in l...",The answer to given question is not present in...,simple,[{'dataset_id': '6762f1b5-2bcc-4062-bff6-e560d...,True
1,How many harvests were conducted in total to m...,"[The dataset entitled ""Grass productivity data...",Three different harvests were conducted in tot...,simple,[{'dataset_id': '6e395915-ab5c-43f4-b4de-c9a3c...,True
2,What specific parameters are recorded for each...,"[The dataset entitled ""UK Environmental Change...",The specific parameters recorded for each tree...,simple,[{'dataset_id': '94aef007-634e-42db-bc52-9aae8...,True
3,What are the specific types of structures and ...,"[The dataset entitled ""Building, infrastructur...",The GIS shapefiles include information about b...,simple,[{'dataset_id': 'a763e254-c249-4934-b0fb-c3b80...,True
4,What are the estimated annual loads of nitroge...,"[The dataset entitled ""Non-agricultural pollut...",The answer to given question is not present in...,simple,[{'dataset_id': 'eb73ca31-7eb9-479c-96be-6063e...,True
...,...,...,...,...,...,...
95,What are the water quality parameters measured...,"[The dataset entitled ""Weekly water quality da...",The water quality parameters measured in the d...,simple,[{'dataset_id': 'cf10ea9a-a249-4074-ac0c-e0c30...,True
96,What are the four scenarios projected for land...,"[The dataset entitled "" Land use maps under th...",The four scenarios projected for land use in t...,simple,[{'dataset_id': 'a94640dc-fe21-4c38-936b-d62df...,True
97,What were the national estimates of Broad Habi...,"[The dataset entitled ""Countryside Survey 1990...",The national estimates of Broad Habitat areas ...,simple,[{'dataset_id': '53ef00f4-e0c5-4095-850e-d4c47...,True
98,What does the 'Generate_data_for_island_model_...,"[The dataset entitled ""Code for generating dat...",The 'Generate_data_for_island_model_weak_selec...,simple,[{'dataset_id': 'aec1e55f-f6ac-4673-9e93-706b4...,True


In [2]:
import pandas as pd
df = pd.read_csv("ragas-testset.csv")
df

,question,contexts,ground_truth,evolution_type,metadata,episode_done
0,What was the average weed abundance across the...,"['The dataset entitled ""Abundance of weeds in ...",The answer to given question is not present in...,simple,[{'dataset_id': '6762f1b5-2bcc-4062-bff6-e560d...,True
1,How many harvests were conducted in total to m...,"['The dataset entitled ""Grass productivity dat...",Three different harvests were conducted in tot...,simple,[{'dataset_id': '6e395915-ab5c-43f4-b4de-c9a3c...,True
2,What specific parameters are recorded for each...,"['The dataset entitled ""UK Environmental Chang...",The specific parameters recorded for each tree...,simple,[{'dataset_id': '94aef007-634e-42db-bc52-9aae8...,True
3,What are the specific types of structures and ...,"['The dataset entitled ""Building, infrastructu...",The GIS shapefiles include information about b...,simple,[{'dataset_id': 'a763e254-c249-4934-b0fb-c3b80...,True
4,What are the estimated annual loads of nitroge...,"['The dataset entitled ""Non-agricultural pollu...",The answer to given question is not present in...,simple,[{'dataset_id': 'eb73ca31-7eb9-479c-96be-6063e...,True
...,...,...,...,...,...,...
95,What are the water quality parameters measured...,"['The dataset entitled ""Weekly water quality d...",The water quality parameters measured in the d...,simple,[{'dataset_id': 'cf10ea9a-a249-4074-ac0c-e0c30...,True
96,What are the four scenarios projected for land...,"['The dataset entitled "" Land use maps under t...",The four scenarios projected for land use in t...,simple,[{'dataset_id': 'a94640dc-fe21-4c38-936b-d62df...,True
97,What were the national estimates of Broad Habi...,"['The dataset entitled ""Countryside Survey 199...",The national estimates of Broad Habitat areas ...,simple,[{'dataset_id': '53ef00f4-e0c5-4095-850e-d4c47...,True
98,What does the 'Generate_data_for_island_model_...,"['The dataset entitled ""Code for generating da...",The 'Generate_data_for_island_model_weak_selec...,simple,[{'dataset_id': 'aec1e55f-f6ac-4673-9e93-706b4...,True


In [4]:
from rag.wrappers import RagPipelineWrapper
import yaml
with open("config.yml", "r") as config_file:
    config = yaml.safe_load(config_file)
pipeline_file = f"{config["pipelines-dir"]}/{config["rag-demo"]["pipeline"]}"
chroma_path = config["vector-db"]["path"]
collection = config["vector-db"]["collection"]
prompt = config["rag-demo"]["prompt"]
rag_pipe = RagPipelineWrapper(
    pipeline_file,
    chroma_path=chroma_path,
    collection=collection,
    prompt=prompt,
)

In [16]:
eval_output = []
for i, row in df.iterrows():
    answer, documents = rag_pipe.query_get_contexts(row["question"])
    result = {
        "question": row["question"],
        "ground_truth": row["ground_truth"],
        "answer": answer,
        "contexts": [doc.content for doc in documents],
    }
    eval_output.append(result)
eval_df = pd.DataFrame(eval_output)

In [17]:
from datasets import Dataset
eval_dataset = Dataset.from_pandas(eval_df)

In [18]:
eval_dataset

Dataset({
    features: ['question', 'ground_truth', 'answer', 'contexts'],
    num_rows: 10
})

In [19]:
from ragas.metrics import (
    answer_relevancy,
    faithfulness,
    context_recall,
    context_precision,
)
from ragas import evaluate
from ragas.run_config import RunConfig
import nest_asyncio
nest_asyncio.apply()
result = evaluate(
    eval_dataset,
    metrics=[
        answer_relevancy,
        faithfulness,
        context_recall,
        context_precision
    ],
    llm=llm,
    embeddings=embeddings,
    is_async=False,
    raise_exceptions=False,
    run_config=RunConfig(max_workers=1)
)
result

Evaluating:   0%|          | 0/40 [00:00<?, ?it/s]

{'answer_relevancy': 0.3347, 'faithfulness': 0.5455, 'context_recall': 0.9500, 'context_precision': 0.7200}